In [ ]:
import pandas as pd

df = pd.read_csv("/content/Combined_UMI_table.txt", sep="\t", index_col=0)

genes_of_interest = ["GAPDH", "PRM1", "PRM2"]
[g for g in genes_of_interest if g in df.index]

['GAPDH', 'PRM1', 'PRM2']

In [ ]:
!pip install scanpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.2/174.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 146.3 MB/s eta 0:00:00


In [ ]:
!pip install scikit-misc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 6.5 MB/s eta 0:00:00


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

df = pd.read_csv("/content/Combined_UMI_table.txt", sep="\t", index_col=0)
adata = sc.AnnData(df.T)

adata.var_names = adata.var_names.astype(str).str.strip()
adata.var_names_make_unique()

adata.layers["counts"] = adata.X.copy()

adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

min_counts = 1000
min_genes = 200
max_pct_mt = 15

adata = adata[
    (adata.obs["total_counts"] >= min_counts)
    & (adata.obs["n_genes_by_counts"] >= min_genes)
    & (adata.obs["pct_counts_mt"] <= max_pct_mt),
    :
]

sc.pp.filter_genes(adata, min_cells=10)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat_v3",
    subset=False
)

panel = set(adata.var_names[adata.var["highly_variable"]]) | set(["PRM1", "PRM2"])
panel = [g for g in panel if g in adata.var_names]

adata = adata[:, panel].copy()

X = pd.DataFrame(
    adata.X.A if hasattr(adata.X, "A") else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

X = (X - X.mean(axis=0)) / X.std(axis=0).replace(0, np.nan)
X = X.fillna(0.0)

expr = X


!wget -q https://resources.aertslab.org/cistarget/tf_lists/allTFs_hg38.txt -O /content/hs_tfs_hg38.txt

tf_list = (
    pd.read_csv("/content/hs_tfs_hg38.txt", header=None)[0]
    .astype(str)
    .str.strip()
    .tolist()
)

tfs_in_data = [t for t in tf_list if t in expr.columns]
print(f"TFs found in dataset after HVG/panel filtering: {len(tfs_in_data)}")


targets = ["PRM1", "PRM2"]


keep_cols = sorted(set(tfs_in_data).union(targets))

keep_cols = [c for c in keep_cols if c in expr.columns]


expr_sub = expr.loc[:, keep_cols].copy()

print(expr.shape, "->", expr_sub.shape)
print("Example of final columns:", expr_sub.columns[:10].tolist())


/usr/local/lib/python3.12/dist-packages/scanpy/preprocessing/_simple.py:293: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number
/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


TFs found in dataset after HVG/panel filtering: 164
(6033, 3001) -> (6033, 166)
Example of final columns: ['AGAP2', 'ALX4', 'ANXA1', 'AR', 'ARX', 'ATF3', 'ATOH8', 'BATF', 'BHLHE40', 'BHLHE41']


In [ ]:
data_fit = expr_sub.copy()
data_fit = data_fit.loc[:, sorted(data_fit.columns)]

In [ ]:
targets = ["PRM1","PRM2"]
tfs = [c for c in data_fit.columns if c not in targets]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_regression

def _zscore_context(mat: np.ndarray, axis: int) -> np.ndarray:
    mu = mat.mean(axis=axis, keepdims=True)
    sigma = mat.std(axis=axis, ddof=1, keepdims=True)
    z = (mat - mu) / (sigma + 1e-12)
    z[z < 0] = 0.0
    return z

def mi_tf_gene_matrix(data: pd.DataFrame, tfs: list[str], genes: list[str], n_neighbors: int = 5) -> pd.DataFrame:
    X = data[tfs].to_numpy()
    mi_mat = np.empty((len(tfs), len(genes)), dtype=float)

    for j, g in enumerate(genes):
        y = data[g].to_numpy()
        mi = mutual_info_regression(X, y, n_neighbors=n_neighbors, random_state=0)
        mi_mat[:, j] = mi

    return pd.DataFrame(mi_mat, index=tfs, columns=genes)

def clr_grn_from_mi(mi_df: pd.DataFrame) -> pd.DataFrame:
    mi = mi_df.to_numpy()

    z_tf = _zscore_context(mi, axis=1)
    z_gene = _zscore_context(mi, axis=0)

    clr = np.sqrt(z_tf**2 + z_gene**2)
    return pd.DataFrame(clr, index=mi_df.index, columns=mi_df.columns)

def clr_grn(data: pd.DataFrame, tfs: list[str], genes: list[str], n_neighbors: int = 5):
    mi_df = mi_tf_gene_matrix(data, tfs, genes, n_neighbors=n_neighbors)
    clr_df = clr_grn_from_mi(mi_df)
    return mi_df, clr_df

def top_tfs_for_target(clr_df: pd.DataFrame, target: str, K: int = 30) -> pd.DataFrame:
    s = clr_df[target].sort_values(ascending=False).head(K)
    return s.reset_index().rename(columns={"index": "TF", target: "CLR_score"})


genes = [c for c in data_fit.columns if c not in tfs]

mi_df, clr_df = clr_grn(data_fit, tfs, genes, n_neighbors=5)

top_prm1_clr = top_tfs_for_target(clr_df, "PRM1", K=30)
top_prm2_clr = top_tfs_for_target(clr_df, "PRM2", K=30)

top_prm1_clr, top_prm2_clr



(         TF  CLR_score
 0     HMGB4   7.692430
 1      YBX1   3.968748
 2     HMGB1   3.862188
 3     H2AFZ   2.820854
 4       RAN   2.696550
 5      CREM   2.191953
 6     RPS4X   2.090823
 7     HMGB2   2.088501
 8      SMC3   2.031371
 9      RBM3   2.029402
 10    HSPA5   1.913314
 11     LSM6   1.266917
 12    TBPL1   1.043638
 13    ANXA1   0.893065
 14     FHL2   0.882945
 15    CEBPD   0.847174
 16  LRRFIP1   0.819925
 17     OSR2   0.801721
 18      ID2   0.778812
 19   CTNNB1   0.776563
 20    HMGA1   0.774446
 21     NFIB   0.743415
 22      JUN   0.737708
 23     ZEB2   0.734418
 24     SSX3   0.717260
 25    HIF3A   0.716325
 26     MYLK   0.711162
 27    SP100   0.710997
 28      MAF   0.709147
 29  GADD45A   0.708692,
         TF  CLR_score
 0    HMGB4   7.131894
 1     YBX1   4.885968
 2    HMGB1   3.930891
 3    H2AFZ   3.047499
 4      RAN   2.684805
 5     CREM   2.559435
 6     SMC3   2.167489
 7    HSPA5   2.165354
 8    HMGB2   2.113240
 9    RPS4X   2.111455
 1